<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myProspects_Upside_Analysis_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance
!pip install ta

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29482 sha256=78ab54cdd2211370283352425e71ec4e0c0ffa1d3de1cc387227c55ecf908bd5
  Stored in directory: /root/.cache/pip/wheels/e3/3a/ee/4955a26c90a4b7deb6d725dc8ec7b8604a7aef44e43a2e8af7
Successfully built ta


In [2]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
import os
project_path = '/content/drive/My Drive/quantvesting'
os.chdir(project_path)

import warnings
warnings.filterwarnings('ignore')

from google.colab import data_table

Mounted at /content/drive


In [3]:
# Import the function from the script
import myProspects_Upside_Analysis_v2

all = 1

# Call the function to get the DataFrame
df_prospects_features = myProspects_Upside_Analysis_v2.get_prospects_features_v2(all)
cols = ['Symbol', 'Close', 'FTT',  'Dev%_200', 'Dev%_PE', 'Spread%', 'Conviction', 'Cyclical','RSI_14', 'RSP', 'Today%', 'FTT%', 'ATH%','Gained%', 'Criteria', 'Strategy', 'Sales_Grwth%', 'Profit_Grwth%', 'Category', 'InFolio', 'MCap', 'MBQ', 'ROE%', 'CFO_2_EBITDA%', 'ROE%/PE']
df_prospects_features = df_prospects_features[cols].drop_duplicates()

Run date time (IST): 2026-11-17 14:14:10
-------------------
get_prospects_features
qualified stocks: 287
with latest results: 287
still star stocks: 112


In [4]:
cols = ['Symbol', 'FTT', 'Dev%_200', 'Dev%_PE', 'Spread%', 'Conviction','Cyclical', 'RSI_14', 'RSP', 'FTT%', 'ATH%','Gained%', 'CumlRnk','ROE%/PE', 'Criteria', 'Strategy', 'Category', 'InFolio']

tmp_df = df_prospects_features
list_x40 = list(tmp_df[tmp_df['MBQ'].str.contains('X40', na=False)]['Symbol'].values)
list_x5k = list(tmp_df[tmp_df['MBQ'].str.contains('X5K', na=False)]['Symbol'].values)
list_x40n = list(tmp_df[tmp_df['MBQ'].str.contains('X40N', na=False)]['Symbol'].values)
list_hc = list(tmp_df[tmp_df['Conviction'].isin(['X-LC', 'H-LC', 'X-MC', 'X-SC'])]['Symbol'].values)
list_ox40 = list(tmp_df[tmp_df['MBQ'].str.contains('OX40', na=False)]['Symbol'].values)
list_ox40n = list(tmp_df[tmp_df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)
#len(set(list_x40 + list_x5k) - set( list_x40n))
stock_list = list(set(list(set(list_x40 + list_x5k + list_hc)- set(list_ox40 + list_ox40n))))

In [5]:
df_tmp = df_prospects_features.copy()

# value ranks (ascending better for deviation from PE and 200DMA)
df_tmp['Rnk_Dev%_200'] = df_tmp['Dev%_200'].rank(method='min', ascending=True)
df_tmp['Rnk_Dev%_PE'] = df_tmp['Dev%_PE'].rank(method='min', ascending=True)
df_tmp['Rnk_ROE%/PE'] = df_tmp['ROE%/PE'].rank(method='max', ascending=False)

# growth ranks
df_tmp['Rnk_Sales_Grwth%'] = df_tmp['Sales_Grwth%'].rank(method='max', ascending=False)
df_tmp['Rnk_Profit_Grwth%'] = df_tmp['Profit_Grwth%'].rank(method='max', ascending=False)

# quality ranks
df_tmp['Rnk_FTT%'] = df_tmp['FTT%'].rank(method='max', ascending=False)
df_tmp['Rnk_MCap'] = df_tmp['MCap'].rank(method='max', ascending=False)
df_tmp['Rnk_ROE%'] = df_tmp['ROE%'].rank(method='max', ascending=False)
df_tmp['Rnk_CFO_2_EBITDA%'] = df_tmp['CFO_2_EBITDA%'].rank(method='max', ascending=False)

# optional Momentum ranks
df_tmp['Rnk_RSI_14'] = df_tmp['RSI_14'].rank(method='max', ascending=False)
df_tmp['Rnk_Gained%'] = df_tmp['Gained%'].rank(method='min', ascending=True)

# category scores
df_tmp['Rank_Value'] = df_tmp[['Rnk_Dev%_200', 'Rnk_Dev%_PE', 'Rnk_ROE%/PE']].sum(axis=1)
df_tmp['Rank_Growth'] = df_tmp[['Rnk_Sales_Grwth%', 'Rnk_Profit_Grwth%']].sum(axis=1)
df_tmp['Rank_Quality'] = df_tmp[['Rnk_FTT%', 'Rnk_MCap', 'Rnk_ROE%', 'Rnk_CFO_2_EBITDA%']].sum(axis=1)
df_tmp['Rank_Momentum'] = df_tmp[['Rnk_Gained%', 'Rnk_RSI_14']].sum(axis=1)

# final cumulative score (you can weigh categories if needed)
df_tmp['Ovrl_Rank'] = df_tmp[['Rank_Value', 'Rank_Growth', 'Rank_Quality', 'Rank_Momentum']].sum(axis=1)

# category-wise rank
df_tmp['Cat_Rank'] = df_tmp.groupby('Conviction')['Ovrl_Rank'].rank(ascending=True, method='min')
priority_map = {'X-LC': 0,'H-LC': 1,'X-MC': 2,'X-SC': 3,'M-LC': 4,'H-MC': 5,'H-SC': 6,'L-LC': 7,'M-MC': 8,'M-SC': 9,'L-MC': 10,'L-SC': 11}
df_tmp['Conviction_Priority'] = df_tmp['Conviction'].map(priority_map)
df_tmp = df_tmp.sort_values(by=['Conviction_Priority', 'Cat_Rank']).reset_index(drop=True)

# final rank
df_tmp['CumlRnk'] = df_tmp.index + 1

# drop intermediate rank columns (optional)
drop_cols = [col for col in df_tmp.columns if col.startswith('Rnk_')]
df_tmp.drop(columns=drop_cols, inplace=True)

# reassign to main dataframe
df_prospects_features = df_tmp[cols]

# display
data_table.DataTable(df_prospects_features.sort_values(by='CumlRnk'), include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
0,NESTLEIND,1540.00,-4.80,-22.66,9.09,X-LC,NC,54.0,53.66,19.38,19.38,11.09,1,1.2,XY25,ATH,FMCG,1.0
1,HINDUNILVR,2922.00,-3.71,-42.16,9.25,X-LC,NC,68.0,77.00,45.19,43.23,11.95,2,1.0,XY25,NTT,FMCG,1.0
2,INFY,1972.00,-1.97,-40.52,7.01,X-LC,NC,52.0,43.90,74.78,64.72,17.14,3,2.1,X40,NTT,IT,1.0
3,TCS,4208.34,2.36,-41.98,7.94,X-LC,NC,52.0,23.69,74.21,74.21,23.16,4,3.1,X40,ATH,IT,1.0
4,BSE,4391.73,12.69,-14.60,5.65,X-LC,NC,67.0,90.94,13.92,13.92,49.75,5,0.9,X40N,ATH,MISC,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282,VALIANTORG,838.00,12.49,NaN,34.80,NaN,SC,37.0,0.35,144.24,439.39,73.60,283,NaN,XR,NTT,CHEMICALS,1.0
283,VEEDOL,2916.05,-2.50,NaN,3.21,NaN,SC,49.0,39.37,113.63,113.63,11.22,284,NaN,XR,ATH,CHEMICALS,NaN
284,VIPIND,489.00,-19.24,NaN,18.41,NaN,NC,44.0,9.06,94.70,194.66,5.28,285,NaN,OX40N,NTT,MISC,NaN
285,WSTCSTPAPR,940.45,20.45,NaN,40.64,NaN,DC,37.0,1.05,40.43,40.43,76.52,286,NaN,XR,ATH,PAPER,NaN


In [6]:
# most interesting sectors
df_tmp = df_prospects_features.groupby('Category').agg(
    Count=('CumlRnk', 'count'),
    Total=('CumlRnk', 'sum'),
    Average=('CumlRnk', 'mean')
).reset_index()
df_tmp['Average'] = round(df_tmp['Average'],2)
df_tmp = df_tmp[df_tmp['Count'] > 3]
df_tmp.sort_values(by='Average')

,Category,Count,Total,Average
23,INSURANCE,8,692,86.50
27,METALS,5,457,91.40
16,FMCG,19,1745,91.84
3,AUTO,12,1138,94.83
7,CEMENT,4,415,103.75
33,PHARMA,18,1873,104.06
12,ELECTRICAL,9,1060,117.78
20,HEALTHCARE,8,961,120.12
21,HOTELS,4,487,121.75
31,PAINTS,5,647,129.40


In [7]:
# module to print date timestamp
def read_csv(csv_file_name):
  csv_file_path = os.path.join(project_path, 'data', csv_file_name)
  df = pd.read_csv(csv_file_path)
  return df

mypps_df = read_csv('myProspectsScrips.csv')

# Suppose this is your updated DataFrame with new cuml_rnk values
cols1 = ['Symbol', 'CumlRnk']
df_tmp = df_prospects_features[cols1]

# Merge the two DataFrames on 'symbol', keeping the CSV structure
mypps_df = mypps_df.merge(df_tmp[['Symbol', 'CumlRnk']], on='Symbol', how='left', suffixes=('', '_new'))

# Replace 'cuml_rnk' with the new value only if it exists in df
mypps_df['CumlRnk'] = mypps_df['CumlRnk_new'].combine_first(mypps_df['CumlRnk'])

# Drop the helper column
mypps_df.drop(columns=['CumlRnk_new'], inplace=True)

# Save back to CSV
#mypps_df.to_csv(project_path+'/data/myProspectsScrips.csv', index=False)

In [8]:
# High quality (X40 + X40N + X5K + X-LC + H-LC + X-MC + X-SC) prospects
df_tmp = df_prospects_features[df_prospects_features['Symbol'].isin(stock_list)]
df_tmp = df_tmp[(df_tmp['FTT%']>20) & (df_tmp['Dev%_PE']<0)]
data_table.DataTable(df_tmp.sort_values(by = 'CumlRnk', ascending=True), include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
1,HINDUNILVR,2922.00,-3.71,-42.16,9.25,X-LC,NC,68.0,77.00,45.19,43.23,11.95,2,1.0,XY25,NTT,FMCG,1.0
2,INFY,1972.00,-1.97,-40.52,7.01,X-LC,NC,52.0,43.90,74.78,64.72,17.14,3,2.1,X40,NTT,IT,1.0
3,TCS,4208.34,2.36,-41.98,7.94,X-LC,NC,52.0,23.69,74.21,74.21,23.16,4,3.1,X40,ATH,IT,1.0
6,VBL,668.62,-5.19,-28.88,8.17,X-LC,NC,54.0,48.08,54.99,54.99,13.38,7,0.4,X40N,ATH,FMCG,1.0
8,ITC,452.00,2.92,-20.80,6.58,X-LC,NC,72.0,83.97,53.56,57.62,15.43,9,1.5,X40,NTT,FMCG,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,RECLTD,446.00,-9.74,-6.47,10.65,M-MC,NC,50.0,32.06,49.74,93.70,5.21,173,3.9,XY25,NTT,FINANCE,1.0
173,IDBI,160.44,-5.88,-32.00,5.64,M-MC,NC,45.0,15.33,102.93,102.93,28.55,174,1.4,X5K,ATH,BANKS,NaN
195,IRB,37.75,-10.45,-37.62,14.72,M-SC,DC,54.0,51.57,110.78,110.78,14.15,196,0.2,X5K,ATH,INFRA,NaN
219,VEDL,772.14,-36.37,-26.33,38.82,L-MC,DC,54.0,60.28,201.11,201.11,13.67,220,8.1,NaN,NaN,MINING,NaN


In [9]:
# X-LC, H-LC & X-MC
df_tmp = df_prospects_features[df_prospects_features['Conviction'].isin(['X-LC','H-LC','X-MC'])].sort_values(by = 'ATH%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
61,TMPV,600.00,-21.73,-528.41,16.83,X-MC,DC,35.0,10.10,130.90,339.67,0.00,62,-1.9,XY24,NTT,AUTO,1.0
62,BAJAJHFL,181.50,-2.71,-39.58,5.61,X-MC,NC,55.0,33.45,119.47,119.47,13.15,63,0.5,X40N,ATH,FINANCE,1.0
53,COLPAL,3628.04,-11.77,-21.86,14.67,X-MC,NC,52.0,32.40,111.97,111.97,8.85,54,2.4,XY25,ATH,FMCG,1.0
56,HAVELLS,2052.62,-3.21,-35.65,10.77,X-MC,SC,67.0,65.16,74.69,74.69,16.51,57,0.4,X40,ATH,ELECTRICAL,1.0
3,TCS,4208.34,2.36,-41.98,7.94,X-LC,NC,52.0,23.69,74.21,74.21,23.16,4,3.1,X40,ATH,IT,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,MCX,3729.60,27.75,-11.77,15.59,X-MC,NC,68.0,92.68,0.43,0.43,92.59,59,1.1,X40N,ATH,MISC,NaN
42,BHARTIARTL,2155.10,14.69,-25.70,7.30,H-LC,SC,74.0,89.37,0.00,0.00,24.22,43,0.5,NaN,ATH,TELECOM,NaN
32,ICICIBANK,1611.20,18.91,3.07,8.30,X-LC,NC,78.0,92.33,0.00,0.00,34.74,33,0.8,X40,ATH,BANKS,NaN
33,INDHOTEL,900.40,28.45,-7.54,19.16,X-LC,DC,76.0,94.77,0.00,0.00,58.40,34,0.3,X40N,ATH,HOTELS,NaN


In [10]:
# x24/x25
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['XY24', 'X25'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
245,DIGITIDE,267.99,-33.43,NaN,27.40,NaN,NC,32.0,3.14,347.32,347.32,0.00,246,NaN,XY24,ATH,IT,NaN
247,EASEMYTRIP,26.40,-14.71,NaN,16.56,NaN,DC,53.0,67.77,342.21,471.36,12.01,248,NaN,XY24,NTT,TRAVEL,1.0
251,GREENPANEL,537.00,-19.15,NaN,25.36,NaN,NC,58.0,68.29,257.07,297.73,20.80,252,NaN,XY24,NTT,MISC,1.0
91,ACC,3906.00,-8.53,19.26,12.80,X-SC,DC,62.0,64.29,213.53,120.36,10.75,92,0.4,XY24,BTT,CEMENT,1.0
77,AWL,485.00,-11.12,-74.63,10.06,X-SC,NC,42.0,7.67,190.25,400.68,1.98,78,0.6,XY24,NTT,FMCG,1.0
124,IRCTC,1269.00,-12.67,-48.45,12.48,H-SC,DC,46.0,28.57,181.62,149.29,3.30,125,1.3,XY24,BTT,TRAVEL,1.0
273,SHALBY,327.00,-21.46,NaN,21.16,NaN,NC,42.0,14.63,170.36,165.96,6.67,274,NaN,XY24,NTT,HEALTHCARE,1.0
61,TMPV,600.00,-21.73,-528.41,16.83,X-MC,DC,35.0,10.10,130.90,339.67,0.00,62,-1.9,XY24,NTT,AUTO,1.0
266,REPCOHOME,880.00,3.41,NaN,10.06,NaN,NC,71.0,91.29,130.61,111.54,27.31,267,NaN,XY24,NTT,FINANCE,1.0
151,BANDHANBNK,400.00,-1.30,8.01,8.67,H-SC,NC,60.0,65.51,128.96,300.96,30.07,152,0.3,XY24,NTT,BANKS,1.0


In [11]:
# X-MC and X-SC prospects
df_tmp = df_prospects_features[df_prospects_features['Conviction'].str.startswith('X-', na=False)]
df_tmp = df_tmp[~df_tmp['Conviction'].isin(['X-LC'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
90,BATAINDIA,2096.00,-6.00,-17.68,6.35,X-SC,SC,50.0,36.24,228.40,214.61,10.08,91,0.2,X40,NTT,FOOTWEAR,1.0
91,ACC,3906.00,-8.53,19.26,12.80,X-SC,DC,62.0,64.29,213.53,120.36,10.75,92,0.4,XY24,BTT,CEMENT,1.0
77,AWL,485.00,-11.12,-74.63,10.06,X-SC,NC,42.0,7.67,190.25,400.68,1.98,78,0.6,XY24,NTT,FMCG,1.0
76,PGHH,17484.41,-22.36,-60.53,22.84,X-SC,NC,48.0,24.39,149.01,149.01,5.76,77,3.9,X40,ATH,FMCG,1.0
81,SANOFI,7552.00,-1.68,-11.88,7.72,X-SC,NC,58.0,89.37,133.10,190.07,19.44,82,2.1,X40,NTT,PHARMA,NaN
61,TMPV,600.00,-21.73,-528.41,16.83,X-MC,DC,35.0,10.10,130.90,339.67,0.00,62,-1.9,XY24,NTT,AUTO,1.0
62,BAJAJHFL,181.50,-2.71,-39.58,5.61,X-MC,NC,55.0,33.45,119.47,119.47,13.15,63,0.5,X40N,ATH,FINANCE,1.0
53,COLPAL,3628.04,-11.77,-21.86,14.67,X-MC,NC,52.0,32.40,111.97,111.97,8.85,54,2.4,XY25,ATH,FMCG,1.0
54,DABUR,735.00,-9.15,-36.14,11.86,X-MC,NC,56.0,46.69,91.26,66.47,8.40,55,0.5,XY24,BTT,FMCG,1.0
88,HONAUT,58192.79,-8.62,-38.05,8.32,X-SC,SC,42.0,15.68,90.86,90.86,16.15,89,0.3,X40N,ATH,ELECTRICAL,1.0


In [12]:
# H-MC and H-SC prospects
df_tmp = df_prospects_features[df_prospects_features['Conviction'].str.startswith('H-', na=False)]
df_tmp = df_tmp[~df_tmp['Conviction'].isin(['H-LC'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
138,METROPOLIS,3102.00,8.45,-9.33,6.83,H-SC,NC,55.0,78.40,444.83,49.81,35.50,139,0.3,XR,NTT,HEALTHCARE,NaN
140,CLEAN,2137.00,-8.39,-41.66,8.16,H-SC,SC,44.0,9.41,202.54,269.05,8.12,141,0.5,XR,NTT,CHEMICALS,NaN
126,INDIAMART,4675.60,-17.67,-60.27,16.56,H-SC,NC,45.0,17.77,201.75,201.75,4.19,127,1.1,AR,ATH,MISC,1.0
124,IRCTC,1269.00,-12.67,-48.45,12.48,H-SC,DC,46.0,28.57,181.62,149.29,3.30,125,1.3,XY24,BTT,TRAVEL,1.0
141,ALKYLAMINE,4520.85,-4.46,-42.24,7.77,H-SC,SC,40.0,5.57,180.66,180.66,31.53,142,0.4,SR,ATH,CHEMICALS,1.0
144,PIIND,4686.44,-37.47,-22.44,26.94,H-SC,SC,26.0,0.70,180.19,180.19,0.00,145,0.4,NaN,ATH,CHEMICALS,NaN
136,TANLA,1903.93,37.32,-12.60,22.46,H-SC,NC,72.0,99.65,167.67,167.67,92.41,137,1.3,AR,ATH,IT,1.0
135,BSOFT,811.90,-6.31,-41.53,9.12,H-SC,NC,46.0,26.48,166.63,166.63,14.59,136,1.0,XR,ATH,IT,1.0
133,TATAELXSI,9161.00,-6.84,-38.66,11.96,H-SC,NC,51.0,23.17,146.45,174.38,24.07,134,0.7,OX40N,NTT,IT,1.0
151,BANDHANBNK,400.00,-1.30,8.01,8.67,H-SC,NC,60.0,65.51,128.96,300.96,30.07,152,0.3,XY24,NTT,BANKS,1.0


In [13]:
# x40
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['X40'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
90,BATAINDIA,2096.00,-6.00,-17.68,6.35,X-SC,SC,50.0,36.24,228.40,214.61,10.08,91,0.2,X40,NTT,FOOTWEAR,1.0
76,PGHH,17484.41,-22.36,-60.53,22.84,X-SC,NC,48.0,24.39,149.01,149.01,5.76,77,3.9,X40,ATH,FMCG,1.0
81,SANOFI,7552.00,-1.68,-11.88,7.72,X-SC,NC,58.0,89.37,133.10,190.07,19.44,82,2.1,X40,NTT,PHARMA,NaN
2,INFY,1972.00,-1.97,-40.52,7.01,X-LC,NC,52.0,43.90,74.78,64.72,17.14,3,2.1,X40,NTT,IT,1.0
56,HAVELLS,2052.62,-3.21,-35.65,10.77,X-MC,SC,67.0,65.16,74.69,74.69,16.51,57,0.4,X40,ATH,ELECTRICAL,1.0
3,TCS,4208.34,2.36,-41.98,7.94,X-LC,NC,52.0,23.69,74.21,74.21,23.16,4,3.1,X40,ATH,IT,1.0
60,ICICILIFE,787.27,-12.62,-54.69,17.34,X-MC,NC,58.0,53.31,73.25,73.25,11.65,61,0.3,X40,ATH,INSURANCE,1.0
80,PAGEIND,50936.07,-14.45,-38.77,5.04,X-SC,DC,29.0,2.79,64.74,64.74,3.35,81,1.2,X40,ATH,APPARELS,1.0
78,PFIZER,6042.10,-15.13,-40.03,15.96,X-SC,NC,47.0,35.19,58.28,58.28,5.68,79,0.9,X40,ATH,PHARMA,NaN
79,GILLETTE,10891.01,-7.63,-40.01,6.13,X-SC,NC,43.0,21.60,53.69,53.69,2.72,80,2.0,X40,ATH,FMCG,1.0


In [14]:
# x40n
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['X40N'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
265,RELAXO,1176.00,-13.76,NaN,13.06,NaN,SC,45.0,27.18,313.21,391.46,20.98,266,NaN,X40N,NTT,FOOTWEAR,1.0
279,TEAMLEASE,3487.00,-22.36,NaN,16.38,NaN,NC,31.0,4.18,255.60,438.52,0.00,280,NaN,X40N,NTT,MISC,NaN
62,BAJAJHFL,181.50,-2.71,-39.58,5.61,X-MC,NC,55.0,33.45,119.47,119.47,13.15,63,0.5,X40N,ATH,FINANCE,1.0
88,HONAUT,58192.79,-8.62,-38.05,8.32,X-SC,SC,42.0,15.68,90.86,90.86,16.15,89,0.3,X40N,ATH,ELECTRICAL,1.0
85,UBL,2211.69,-13.29,-42.49,13.72,X-SC,NC,48.0,29.27,84.97,84.97,5.24,86,0.1,X40N,ATH,FMCG,NaN
89,LALPATHLAB,3545.00,19.09,-7.61,16.01,X-SC,NC,55.0,70.56,79.54,2.74,52.15,90,0.4,X40N,NTT,HEALTHCARE,NaN
83,ASTRAZEN,10427.12,-20.89,-17.73,20.15,X-SC,NC,46.0,42.33,69.02,69.02,7.86,84,0.3,X40N,ATH,PHARMA,NaN
55,GODREJCP,1452.16,-11.03,-22.63,15.48,X-MC,NC,62.0,62.02,62.90,62.90,9.64,56,0.3,X40N,ATH,FMCG,1.0
9,DMART,5391.45,-12.29,-29.34,15.75,X-LC,NC,54.0,56.79,56.17,56.17,10.16,10,0.2,X40N,ATH,FMCG,1.0
6,VBL,668.62,-5.19,-28.88,8.17,X-LC,NC,54.0,48.08,54.99,54.99,13.38,7,0.4,X40N,ATH,FMCG,1.0


In [15]:
# xr, sr, ar
df_tmp = df_prospects_features[df_prospects_features['Criteria'].isin(['XR','SR', 'AR'])].sort_values(by = 'FTT%', ascending=False)
data_table.DataTable(df_tmp, include_index=False)

,Symbol,FTT,Dev%_200,Dev%_PE,Spread%,Conviction,Cyclical,RSI_14,RSP,FTT%,ATH%,Gained%,CumlRnk,ROE%/PE,Criteria,Strategy,Category,InFolio
236,BAJAJHIND,442.94,-0.50,NaN,9.98,NaN,DC,41.0,5.23,2373.14,2373.14,19.96,237,NaN,XR,ATH,SUGAR,NaN
231,ALOKINDS,79.81,-32.75,NaN,33.97,NaN,DC,44.0,27.87,888.97,888.97,14.47,232,NaN,AR,ATH,TEXTILES,NaN
225,CENTRALBK,183.18,-1.09,-49.25,8.11,L-SC,NC,70.0,59.93,461.21,461.21,13.61,226,2.0,XR,ATH,BANKS,NaN
138,METROPOLIS,3102.00,8.45,-9.33,6.83,H-SC,NC,55.0,78.40,444.83,49.81,35.50,139,0.3,XR,NTT,HEALTHCARE,NaN
268,ROUTE,2170.75,-12.97,NaN,11.29,NaN,NC,44.0,31.01,400.87,400.87,6.43,269,NaN,SR,ATH,IT,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,NEOGEN,2856.30,47.36,92.65,28.07,M-SC,SC,66.0,81.53,3.04,3.04,183.04,218,0.0,AR,ATH,CHEMICALS,NaN
210,CUB,238.48,11.61,18.87,7.96,M-SC,NC,58.0,38.33,2.23,2.23,35.61,211,0.9,AR,ATH,BANKS,NaN
40,JSWSTEEL,1357.10,6.50,-66.09,3.51,H-LC,DC,60.0,69.16,1.95,1.95,24.05,41,0.9,AR,ATH,STEEL,NaN
243,DCBBANK,233.01,19.80,NaN,10.09,NaN,NC,66.0,74.22,1.83,1.83,45.74,244,NaN,AR,ATH,BANKS,NaN


In [16]:
# conviction-wise prospects count
df_prospects_features['Conviction'].value_counts()

,count
Conviction,
X-LC,36
M-SC,33
H-SC,32
X-MC,23
M-MC,22
X-SC,19
H-LC,17
H-MC,15
M-LC,14
